# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]


In [2]:

# Compute similarities

#YOUR CODE HERE

for sent1, sent2 in sentence_pairs:
  # Step 1: Encode the two sentences to get their embeddings (their number lists)
  embeddings = model.encode([sent1, sent2])

  # Step 2: Calculate the cosine similarity between the first and second embedding
  similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
  
  # Step 3: Print the sentences and their similarity score
  print(f"Sentence 1: {sent1}")
  print(f"Sentence 2: {sent2}")
  print(f"Similarity Score: {similarity:.4f}") # :.4f formats the number to 4 decimal places
  print("---")


Sentence 1: A dog is playing in the park.
Sentence 2: A dog is running in a field.
Similarity Score: 0.5220
---
Sentence 1: I love pizza.
Sentence 2: I enjoy ice cream.
Similarity Score: 0.5281
---
Sentence 1: What is AI?
Sentence 2: How does a computer learn?
Similarity Score: 0.3194
---


### Questions:
- Which sentence pairs are the most semantically similar? Why?


**1. Which sentence pairs are the most semantically similar? Why?**

Looking at the scores:
*   Dog playing / Dog running: **0.5220**
*   Love pizza / Enjoy ice cream: **0.5281**
*   What is AI? / Computer learn: **0.3194**

The most semantically similar pair is **"I love pizza."** and **"I enjoy ice cream."** with a score of **0.5281**.

**Why?**
Even though a dog playing and a dog running seems more similar at first glance, the embedding model is looking at the *meaning of the words*.
*   "Love" and "enjoy" are very similar emotions or feelings.
*   "Pizza" and "ice cream" are both types of popular snack foods.
So the overall structure and meaning of the sentences are very similar: `[Person] + [positive feeling] + [popular food]`.

The dog sentences have a high score (0.5220) for a similar reason: `[A dog] + [is] + [an activity] + [in a place]`. The main difference is the activity ("playing" vs. "running") and the place ("park" vs. "field"), which is why the score is slightly lower.


- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


**2. Can you think of cases where cosine similarity might fail to capture true semantic meaning?**

Yes, absolutely. Cosine similarity looks at the overall direction of the number vectors but can sometimes miss specific, important details. Here are two common cases where it can fail:

**Case 1: Opposites (Negation)**
*   **Sentence A:** "I love this movie, it's fantastic!"
*   **Sentence B:** "I do not love this movie, it's terrible."
*   **Why it might fail:** The sentences share almost all the same words ("I", "love", "this", "movie"). The model might give them a high similarity score because the *word vectors* are close together. However, the true meaning is completely opposite because of the word "not". The model might not fully understand how negation flips the entire meaning.

**Case 2: Specificity vs. Generality**
*   **Sentence A:** "The vehicle had a faulty brake pedal."
*   **Sentence B:** "The car had a problem with its brakes."
*   **Why it might fail:** These sentences mean almost the same thing ("vehicle" vs. "car", "faulty brake pedal" vs. "problem with brakes"). However, if the model was trained on data where "vehicle" and "car" are used in very different contexts, it might give them a surprisingly low score, failing to see they are semantically equivalent.

**Case 3: Cultural or Contextual Meaning**
*   **Sentence A:** "That party was so lit!" (slang for "great")
*   **Sentence B:** "The building was on fire." (literal fire)
*   **Why it might fail:** The word "lit" is the key word in both. Without understanding modern slang, the model might think these sentences are very similar because they both contain a word related to light/fire, completely missing the intended meaning.

In summary, cosine similarity is a powerful tool, but it's a mathematical measure. It doesn't truly "understand" language like a human does.

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [4]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]


In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')
# Encode documents


document_embeddings = model.encode(documents)
#YOUR CODE HERE

In [9]:
print(f"Shape of embeddings: {document_embeddings.shape}")

Shape of embeddings: (6, 384)


In [ ]:
# Perform KMeans clustering
# We choose to find 2 clusters. We'll discuss how to choose this number later.
num_clusters = 2
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')
kmeans.fit(document_embeddings)

# Get the cluster assignment for each document
cluster_assignments = kmeans.labels_

# pred = kmeans.predict(X)

**These two methods do the exact same thing for the data you used to train (`X` or `document_embeddings`).**

*   `kmeans.labels_` is a direct shortcut. After you `.fit()`, the labels are already calculated and stored in this attribute.
*   `kmeans.predict(X)` is a function you would use **on new, unseen data**. It takes the cluster centers that were learned during `.fit()` and assigns the new data to the closest cluster.

### Real-World Example:

Imagine you are a teacher (`KMeans` algorithm) and you have a class of 6 students (`documents`).

1.  **`.fit()` (Training):** You look at all 6 students and decide to form 2 project groups (`clusters`) based on their skills.
2.  **`.labels_`:** Right after forming the groups, you look at your list to see which student is in which group. **This is what we did.**
3.  **`.predict()` (Prediction):** A **new student** joins the class later. You now look at this new student's skills and use your existing group rules to decide which of the two project groups they should join.

So, for our task, where we are *only* clustering the data we have, using `kmeans.labels_` is the perfect and most direct way to get the results.

**In summary: You did it correctly!** Using `.predict()` on the same data you trained with is unnecessary because the answer is already available instantly in `.labels_`.

Knowing about `.predict()` is crucial for when onces want to use the clustering model on new data later. For example, with a new question like "What is the capital of Spain?", you encode it and use `kmeans.predict([new_embedding])` to see which cluster it belongs to!

In [12]:
cluster_assignments

array([0, 0, 0, 1, 1, 1], dtype=int32)

In [ ]:
# Print cluster assignments
for cluster_id in range(num_clusters):
    print(f"\n--- Cluster #{cluster_id} ---")
    # Find the indices of all documents that belong to the current cluster
    indices = [i for i, x in enumerate(cluster_assignments) if x == cluster_id]
    # Print each document in this cluster
    for index in indices:
        print(f"• {documents[index]}")


--- Cluster #0 ---
• What is the capital of France?
• How do I bake a chocolate cake?
• What is the distance between Earth and Mars?

--- Cluster #1 ---
• How do I change a flat tire on a car?
• What is the best way to learn Python?
• How do I fix a leaky faucet?


### Questions:

- How many clusters make the most sense? Why?

- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

**1. How many clusters make the most sense? Why?**

Based on the output, **2 clusters** seems like a reasonable choice. The algorithm found a natural split in the data. We asked for 2 groups (`n_clusters=2`) and it gave us two groups that we can try to make sense of.

*We could try 3 clusters to see if it finds a more precise grouping, but 2 is a good starting point.*


**2. Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?**

Yes, the clusters are semantically meaningful! The sentences in each group share a common underlying theme. Here is how we can interpret them:

*   **Cluster #0 Theme: `Factual & Trivia Questions`**
    *   `"What is the capital of France?"` (Geography fact)
    *   `"What is the distance between Earth and Mars?"` (Science fact)
    *   `"How do I bake a chocolate cake?"` (This one is interesting! It's a "how-to", but the model might have grouped it here because it's a very common, almost "general knowledge" question compared to the others. It's asking for a *fact* about baking - the recipe.)

*   **Cluster #1 Theme: `Practical & Self-Improvement How-To Guides`**
    *   `"How do I change a flat tire on a car?"` (Practical skill)
    *   `"What is the best way to learn Python?"` (Self-improvement skill)
    *   `"How do I fix a leaky faucet?"` (Practical skill)

**Why is "bake a cake" in Cluster #0?**
This is a great example of how the model "thinks". It doesn't see words like "fact" or "how-to". It only sees the relationships between the words in their numeric form.

*   "What is the capital..." and "What is the distance..." both start with `"What is the..."`. This is a very strong signal.
*   "How do I bake..." might be semantically closer to "What is the..." questions than to the other "How do I..." questions about car tires and faucets. The concept of "baking a cake" might be numerically closer to "France" and "Mars" in the model's embedding space than to "car" or "faucet".


---

### Summary of the Results:

The algorithm found two distinct groups:
1.  **Cluster 0:** Questions about general knowledge and facts.
2.  **Cluster 1:** Questions about learning practical skills.

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [ ]:
import numpy as np

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

from sentence_transformers import SentenceTransformer

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')


In [17]:
# Compute document embeddings
document_embeddings = model.encode(documents)
print(f"Shape of embeddings: {document_embeddings.shape}")
print(f"Number of documents: {len(documents)}")

#YOUR CODE HERE

Shape of embeddings: (10, 384)
Number of documents: 10


In [18]:
# Create the search function
#This function should encode the user query and return the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    # YOUR CODE HERE

    # 1. Encode the user query to get its embedding
    query_embedding = model.encode([query])

    # 2. Calculate similarity between the query and every document
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    
    # 3. Get the indices of the top_n most similar documents
    # argsort sorts the list of similarities and returns the indices (positions) of the sorted values
    top_indices = np.argsort(similarities)[::-1][:top_n]

    # 4. Print the results
    print(f"Query: '{query}'\n")
    print("Top most similar documents:")
    print("-" * 50)
    for rank, index in enumerate(top_indices):
        # Print the rank (1st, 2nd), the document text, and its similarity score
        print(f"{rank+1}. {documents[index]} (Similarity: {similarities[index]:.4f})")
    
    # (Optional) Return the results if another function needs to use them
    return top_indices

In [20]:
# Test the search function
query = "Explain programming languages."
semantic_search(query, documents, document_embeddings)

Query: 'Explain programming languages.'

Top most similar documents:
--------------------------------------------------
1. What is quantum computing? (Similarity: 0.4352)
2. What is the best way to learn Python? (Similarity: 0.3188)
3. How do I build a mobile app? (Similarity: 0.1104)
4. How do I set up a local server? (Similarity: 0.0911)
5. What are the best travel destinations in Europe? (Similarity: 0.0906)


array([8, 4, 9, 7, 6])

In [21]:
# Test the search function
query = "I want to go to space"
semantic_search(query, documents, document_embeddings)

Query: 'I want to go to space'

Top most similar documents:
--------------------------------------------------
1. What is the distance between Earth and Mars? (Similarity: 0.3095)
2. What are the best travel destinations in Europe? (Similarity: 0.3000)
3. What is the best way to learn Python? (Similarity: 0.1352)
4. What is the capital of France? (Similarity: 0.1140)
5. How do I bake a chocolate cake? (Similarity: 0.1095)


array([2, 6, 4, 0, 1])

### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

**1. What are the top-ranked results for the given queries?**

For the query **"Explain programming languages."**, the top results were:
1.  **What is quantum computing?** (Similarity: 0.4352)
2.  **What is the best way to learn Python?** (Similarity: 0.3188)
3.  How do I build a mobile app? (Similarity: 0.1104)
4.  How do I set up a local server? (Similarity: 0.0911)
5.  What are the best travel destinations in Europe? (Similarity: 0.0906)

**2. How can you improve the ranking explanation for users?**

The results are good, but not perfect. The top result is about quantum computing, which is related to computer science but not directly about "programming languages". Here are ways to improve the explanation and the ranking itself:

**A) Improve the Explanation (Make it clearer for the user):**
*   **Add a Threshold:** Don't show results with very low similarity scores. The last three results (about mobile apps, servers, and travel) have very low scores (around 0.09). This suggests they are not really related. You could set a rule: `if similarity < 0.2: don't show it`. This would clean up the results and only show the top 2, which are much more relevant.
*   **Explain the Connection:** You could add a brief reason *why* a result was found.
    *   For result #1: *"This result is about an advanced field of computer science."*
    *   For result #2: *"Python is a specific programming language."*
    This helps the user understand the semantic connection, even if it's not a perfect keyword match.

**B) Improve the Ranking (Get better results):**
*   **Use a Different Model:** The `'all-MiniLM-L6-v2'` model is good for speed and general use, but larger models like `'all-mpnet-base-v2'` are often better at capturing nuanced meaning. They are slower but more accurate.
*   **Pre-process the Text:** Cleaning the text (removing question marks, stop words like "what", "how") can sometimes help the model focus on the key words, though modern embedding models are already very good at this.

**3. Why did "quantum computing" score higher than "learn Python"?**

This is a fascinating result that shows how the model "thinks". The word **"what is"** in the query `"Explain programming languages."` is a very strong signal.

*   The model likely learned that phrases beginning with **"What is"** are often broad, conceptual, explanatory questions.
*   `"What is quantum computing?"` is *exactly* that type of question.
*   `"What is the best way to learn Python?"` is a "how-to" question disguised as a "what is" question. It's more about a method than a definition.

The model matched the *style and structure* of the question very well, finding the most conceptually similar question in the list.

**Summary:** The search engine successfully found the two most relevant documents (quantum computing and learning Python) and ranked them based on its understanding of the sentence structure and meaning. To make it more useful for a user, you would add a similarity threshold to filter out weak results.